# DAG Structure Exploration

This notebook explores the TokenDAG data structure and different DAG templates for reasoning.
Use this to:
- Visualize different DAG structures
- Understand topological levels and unmasking schedules
- Compare structural properties across templates
- Prototype new DAG templates

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
import matplotlib.pyplot as plt

from dllm_reason.graph.dag import TokenDAG
from dllm_reason.graph.templates import (
    chain_of_thought_dag, skeleton_then_detail_dag,
    bidirectional_dag, interleaved_dag, random_dag,
)
from dllm_reason.graph.viz import draw_dag, draw_unmasking_timeline
from dllm_reason.eval.dag_analysis import analyze_dag, compare_dags, plot_level_distribution

SEQ_LEN = 32
print('Imports OK')

## 1. Basic DAG Operations

In [ ]:
# Empty DAG: all positions independent
empty = TokenDAG.empty(SEQ_LEN)
print('Empty DAG:', empty)

# Linear chain: 0 -> 1 -> 2 -> ...
chain = TokenDAG.linear_chain(SEQ_LEN)
print('Linear chain:', chain)

# Custom DAG from edges
custom = TokenDAG.from_edges(SEQ_LEN, [(0, 4), (0, 8), (4, 12), (8, 12), (12, 16)])
print('Custom DAG:', custom)
print('Is valid DAG?', custom.is_valid())

In [ ]:
# Topological levels
levels = chain_of_thought_dag(SEQ_LEN, num_steps=4).topological_levels()
print(f'CoT DAG has {len(levels)} levels:')
for i, level in enumerate(levels):
    print(f'  Level {i}: {level}')

In [ ]:
# Ready positions demo
dag = chain_of_thought_dag(SEQ_LEN, num_steps=4)
is_unmasked = torch.zeros(SEQ_LEN, dtype=torch.bool)

print('Initially ready:', dag.ready_positions(is_unmasked).nonzero().squeeze().tolist())

# Unmask first 8 positions
is_unmasked[:8] = True
print('After unmasking [0..7], ready:', dag.ready_positions(is_unmasked).nonzero().squeeze().tolist())

## 2. Template DAGs

In [ ]:
dags = {
    'Empty': TokenDAG.empty(SEQ_LEN),
    'Linear': TokenDAG.linear_chain(SEQ_LEN),
    'CoT (4 steps)': chain_of_thought_dag(SEQ_LEN, 4),
    'Skeleton': skeleton_then_detail_dag(SEQ_LEN,
                    skeleton_positions=list(range(0, SEQ_LEN, 3)),
                    detail_positions=list(range(1, SEQ_LEN, 3))),
    'Bidirectional': bidirectional_dag(SEQ_LEN, 4),
    'Interleaved': interleaved_dag(SEQ_LEN, 3),
}

# Draw all
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, (name, dag) in zip(axes.flatten(), dags.items()):
    import networkx as nx
    G = dag.to_networkx()
    levels = dag.topological_levels()
    for i, lv in enumerate(levels):
        for node in lv:
            G.nodes[node]['level'] = i
    try:
        pos = nx.multipartite_layout(G, subset_key='level')
    except:
        pos = nx.spring_layout(G)
    nx.draw(G, pos, ax=ax, node_size=100, arrows=True,
            node_color='steelblue', alpha=0.8)
    ax.set_title(f'{name}\n(depth={dag.depth()}, edges={dag.num_edges()})')

plt.suptitle('DAG Template Structures', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Structural Comparison

In [ ]:
stats = compare_dags(dags)

In [ ]:
fig = plot_level_distribution(dags, figsize=(16, 4))
plt.show()

## 4. Unmasking Schedule Visualization

In [ ]:
from dllm_reason.eval.dag_analysis import plot_unmasking_heatmap

fig, axes = plt.subplots(3, 2, figsize=(14, 8))
for ax, (name, dag) in zip(axes.flatten(), dags.items()):
    schedule = dag.to_mask_schedule(16)
    step_map = torch.full((SEQ_LEN,), -1)
    for step, positions in enumerate(schedule):
        for pos in positions:
            if step_map[pos] == -1:
                step_map[pos] = step
    im = ax.imshow(step_map.unsqueeze(0).numpy(), aspect='auto',
                   cmap='plasma', vmin=0, vmax=16)
    ax.set_title(name)
    ax.set_yticks([])
    ax.set_xlabel('Token Position')
    plt.colorbar(im, ax=ax, label='Step')

plt.suptitle('Unmasking Timeline (16 steps)', fontsize=13)
plt.tight_layout()
plt.show()

## 5. DAG Mutation and Validity

In [ ]:
from dllm_reason.graph.constraints import topological_mutation

base = chain_of_thought_dag(SEQ_LEN, 4)
print('Base:', base)

# Mutate: add 2 edges, remove 1
mutated = topological_mutation(base, num_add=2, num_remove=1)
print('Mutated:', mutated)
print('Still valid?', mutated.is_valid())